In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_numopt
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import *
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from train_loop import train_loop

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
class Net(nn.Module):
    def __init__(self, input_size, device="cpu"):
        super().__init__()
        self.f1 = nn.Linear(input_size, 10, device=device)
        self.f2 = nn.Linear(10, 20, device=device)
        self.f3 = nn.Linear(20, 20, device=device)
        self.f4 = nn.Linear(20, 10, device=device)
        self.f5 = nn.Linear(10, 1, device=device)

        self.activation = nn.ReLU()
        # self.activation = nn.Sigmoid()
        # self.activation = nn.Tanh()

    def forward(self, x):
        x = self.activation(self.f1(x))
        x = self.activation(self.f2(x))
        x = self.activation(self.f3(x))
        x = self.activation(self.f4(x))
        x = self.f5(x)

        return x

In [4]:
# X, y = load_diabetes(return_X_y=True, scaled=False)
# X, y = make_regression(n_samples=1000, n_features=100)
# X, y = make_friedman1(n_samples=10, noise=1e-2)
# X, y = make_friedman1(n_samples=100, noise=1e-2)
# X, y = make_friedman1(n_samples=1000, noise=1e-2)
X, y = make_friedman1(n_samples=1000, noise=1e-2)

X_scaler = MinMaxScaler()
X = X_scaler.fit_transform(X)
X = torch.Tensor(X).to(device)

y_scaler = MinMaxScaler()
y = y_scaler.fit_transform(y.reshape((-1, 1)))
y = torch.Tensor(y).to(device)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch_data = TensorDataset(X_train, y_train)
data_loader = DataLoader(torch_data, batch_size=10000)

torch.Size([800, 10]) torch.Size([800, 1])
torch.Size([200, 10]) torch.Size([200, 1])


In [5]:
model = Net(input_size=X.shape[1], device=device)
loss_fn = nn.MSELoss()
batch_size = 100
# batch_size = None

# opt = torch_numopt.NewtonRaphson(model=model, lr=0.01, line_search_method="const", batch_size=batch_size)
opt = torch_numopt.NewtonRaphson(model=model, lr=1, c1=1e-4, tau=0.1, line_search_method="backtrack", line_search_cond="armijo", batch_size=batch_size)
# opt = torch_numopt.NewtonRaphson(model=model, lr=1, c1=1e-4, tau=0.5, line_search_method="backtrack", line_search_cond="wolfe", batch_size=batch_size)
# opt = torch_numopt.NewtonRaphson(model=model, lr=1, c1=1e-4, tau=0.5, line_search_method="backtrack", line_search_cond="strong-wolfe", batch_size=batch_size)
# opt = torch_numopt.NewtonRaphson(model=model, lr=1, c1=1e-4, tau=0.5, line_search_method="backtrack", line_search_cond="goldstein", batch_size=batch_size)
# opt = optim.Adam(model.parameters())

model, loss_history = train_loop(
    model,
    loss_fn,
    opt,
    data_loader,
    epochs=1000,
    max_patience=50
)

epoch:  0, loss: 0.09909110516309738
epoch:  1, loss: 0.0901050716638565
epoch:  2, loss: 0.07670102268457413
epoch:  3, loss: 0.05713753402233124
epoch:  4, loss: 0.04953878000378609
epoch:  5, loss: 0.046818021684885025
epoch:  6, loss: 0.04061942175030708
epoch:  7, loss: 0.0239630788564682
epoch:  8, loss: 0.012226279824972153
epoch:  9, loss: 0.011878006160259247
epoch:  10, loss: 0.010298691689968109
epoch:  11, loss: 0.008226320147514343
epoch:  12, loss: 0.006880881730467081
epoch:  13, loss: 0.006058520171791315
epoch:  14, loss: 0.0053202370181679726
epoch:  15, loss: 0.004873909987509251
epoch:  16, loss: 0.0044884770177304745
epoch:  17, loss: 0.004217580892145634
epoch:  18, loss: 0.003970828838646412
epoch:  19, loss: 0.003770183539018035
epoch:  20, loss: 0.0035844373051077127
epoch:  21, loss: 0.0034424462355673313
epoch:  22, loss: 0.003287771949544549
epoch:  23, loss: 0.0031466283835470676
epoch:  24, loss: 0.0030287611298263073
epoch:  25, loss: 0.002920608501881361

In [6]:
pred_train = model.forward(X_train).cpu().detach()
pred_test = model.forward(X_test).cpu().detach()
print(f"Train metrics: R2 = {r2_score(pred_train, y_train.cpu())}")
print(f"Test metrics:  R2 = {r2_score(pred_test, y_test.cpu())}")

Train metrics: R2 = 0.9730591177940369
Test metrics:  R2 = 0.724632978439331
